<a href="https://colab.research.google.com/github/Rds1007/SQL_BigDataInterview/blob/main/avg_session_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark=SparkSession.builder.appName("avg session time").getOrCreate()

In [4]:
df=spark.read.csv("/content/sample_data/facebook_web_log.csv",header=True,inferSchema=True)


+-------+-------------------+-----------+
|user_id|          timestamp|     action|
+-------+-------------------+-----------+
|      0|2019-04-25 13:30:15|  page_load|
|      0|2019-04-25 13:30:18|  page_load|
|      0|2019-04-25 13:30:40|scroll_down|
|      0|2019-04-25 13:30:45|  scroll_up|
|      0|2019-04-25 13:31:10|scroll_down|
|      0|2019-04-25 13:31:25|scroll_down|
|      0|2019-04-25 13:31:40|  page_exit|
|      1|2019-04-25 13:40:00|  page_load|
|      1|2019-04-25 13:40:10|scroll_down|
|      1|2019-04-25 13:40:15|scroll_down|
|      1|2019-04-25 13:40:20|scroll_down|
|      1|2019-04-25 13:40:25|scroll_down|
|      1|2019-04-25 13:40:30|scroll_down|
|      1|2019-04-25 13:40:35|  page_exit|
|      2|2019-04-25 13:41:21|  page_load|
|      2|2019-04-25 13:41:30|scroll_down|
|      2|2019-04-25 13:41:35|scroll_down|
|      2|2019-04-25 13:41:40|  scroll_up|
|      1|2019-04-26 11:15:00|  page_load|
|      1|2019-04-26 11:15:10|scroll_down|
+-------+-------------------+-----

In [5]:
df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- action: string (nullable = true)



In [15]:
df_enter=df.filter(F.col('action') == 'page_load').withColumn("Cal_Day",F.to_date("timestamp",'yyyy-MM-dd')).groupBy("user_id","Cal_Day").agg(F.min("timestamp").alias("enter_time"))
df_enter.show()

+-------+----------+-------------------+
|user_id|   Cal_Day|         enter_time|
+-------+----------+-------------------+
|      1|2019-04-25|2019-04-25 13:40:00|
|      1|2019-04-26|2019-04-26 11:15:00|
|      2|2019-04-25|2019-04-25 13:41:21|
|      0|2019-04-25|2019-04-25 13:30:00|
|      0|2019-04-28|2019-04-28 14:30:10|
+-------+----------+-------------------+



In [16]:
df_exit=df.filter(F.col("action")=='page_exit').withColumn("Cal_Day",F.to_date("timestamp",'yyyy-MM-dd')).groupBy("user_id","Cal_Day").agg(F.min("timestamp").alias("exit_time"))
df_exit.show()

+-------+----------+-------------------+
|user_id|   Cal_Day|          exit_time|
+-------+----------+-------------------+
|      1|2019-04-25|2019-04-25 13:40:35|
|      1|2019-04-26|2019-04-26 11:15:35|
|      0|2019-04-25|2019-04-25 13:30:40|
|      0|2019-04-28|2019-04-28 15:31:40|
+-------+----------+-------------------+



In [17]:
df_joined=df_enter.join(df_exit,["user_id","Cal_Day"],"inner").withColumn("session_time",F.unix_timestamp("exit_time")-F.unix_timestamp("enter_time"))
df_joined.show()

+-------+----------+-------------------+-------------------+------------+
|user_id|   Cal_Day|         enter_time|          exit_time|session_time|
+-------+----------+-------------------+-------------------+------------+
|      1|2019-04-25|2019-04-25 13:40:00|2019-04-25 13:40:35|          35|
|      1|2019-04-26|2019-04-26 11:15:00|2019-04-26 11:15:35|          35|
|      0|2019-04-25|2019-04-25 13:30:00|2019-04-25 13:30:40|          40|
|      0|2019-04-28|2019-04-28 14:30:10|2019-04-28 15:31:40|        3690|
+-------+----------+-------------------+-------------------+------------+



In [18]:
df_average=df_joined.groupBy("user_id").agg(F.avg("session_time").alias("average_session_time"))
df_average.show()

+-------+--------------------+
|user_id|average_session_time|
+-------+--------------------+
|      1|                35.0|
|      0|              1865.0|
+-------+--------------------+

